# Draft Analysis Uncertainty Quantification

## Confidence Intervals for VOR and Draft Grades

This notebook visualizes the **uncertainty** in our draft analysis metrics:

### Why Uncertainty Matters:
- Prevents **false precision** (reporting "73.2" when real range is 65-80)
- Shows **statistical rigor** (not just point estimates)
- Helps **risk assessment** (wide CI = high uncertainty = risky pick)
- Makes analysis **publication-ready** (academic standard)

### What We'll Visualize:
1. VOR confidence intervals by player
2. Grade score uncertainty ranges
3. Distribution of grades with error bars
4. Comparison of high vs low uncertainty picks

Let's explore the uncertainty!

In [ ]:
import duckdb
import plotly.graph_objects as go
import plotly.express as px

conn = duckdb.connect('../data/warehouse.duckdb', read_only=True)
print("✅ Connected to warehouse")

## 1. Load Draft Performance with Uncertainty Metrics

In [ ]:
# Load draft performance with uncertainty data
draft = conn.execute("""
    SELECT 
        pick_no,
        round,
        player_name,
        position,
        manager_name,
        games_played,
        risk_adjusted_scarcity_vor,
        risk_adjusted_vor_lower_bound,
        risk_adjusted_vor_upper_bound,
        vor_uncertainty_range,
        vor_cv,
        grade_score,
        grade_score_lower_bound,
        grade_score_upper_bound,
        grade_score_uncertainty,
        pick_grade,
        risk_tier,
        consistency_tier
    FROM fct_draft_performance
    WHERE games_played > 0
    ORDER BY pick_no
""").df()

print(f"Loaded {len(draft)} picks with active players")
print("\nSample uncertainty metrics:")
print(draft[['player_name', 'risk_adjusted_scarcity_vor', 'vor_uncertainty_range', 'grade_score', 'grade_score_uncertainty']].head(10))

## 2. VOR Confidence Intervals (Top 30 Picks)

Visualize VOR with error bars showing ±1 standard deviation.

In [ ]:
# Focus on top 30 picks for clarity
top_picks = draft[draft['pick_no'] <= 30].copy()

fig = go.Figure()

# Add error bars for each pick
for _, row in top_picks.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['pick_no']],
        y=[row['risk_adjusted_scarcity_vor']],
        error_y=dict(
            type='data',
            symmetric=False,
            array=[row['risk_adjusted_vor_upper_bound'] - row['risk_adjusted_scarcity_vor']],
            arrayminus=[row['risk_adjusted_scarcity_vor'] - row['risk_adjusted_vor_lower_bound']],
            color='lightblue',
            thickness=2,
            width=4
        ),
        mode='markers',
        marker=dict(size=10, color=px.colors.qualitative.Set2[['QB', 'RB', 'WR', 'TE'].index(row['position'])]),
        name=row['player_name'],
        hovertemplate=f"<b>{row['player_name']}</b><br>" +
                      f"Pick #{row['pick_no']}<br>" +
                      f"VOR: {row['risk_adjusted_scarcity_vor']:.1f}<br>" +
                      f"Range: {row['risk_adjusted_vor_lower_bound']:.1f} - {row['risk_adjusted_vor_upper_bound']:.1f}<br>" +
                      f"Uncertainty: ±{row['vor_uncertainty_range']/2:.1f}<extra></extra>",
        showlegend=False
    ))

fig.update_layout(
    title='VOR with Confidence Intervals (Top 30 Picks)',
    xaxis_title='Draft Pick Number',
    yaxis_title='Risk-Adjusted VOR',
    height=600,
    template='plotly_white',
    hovermode='closest'
)

fig.show()

print("\nMost Certain Picks (Narrow CI):")
print("=" * 60)
print(top_picks.nsmallest(5, 'vor_uncertainty_range')[['pick_no', 'player_name', 'position', 'risk_adjusted_scarcity_vor', 'vor_uncertainty_range', 'consistency_tier']])

print("\n\nMost Uncertain Picks (Wide CI):")
print("=" * 60)
print(top_picks.nlargest(5, 'vor_uncertainty_range')[['pick_no', 'player_name', 'position', 'risk_adjusted_scarcity_vor', 'vor_uncertainty_range', 'consistency_tier']])

## 3. Grade Score Distributions by Round

Show grade scores with uncertainty ranges (fan chart style).

In [ ]:
# Group by round and calculate stats
round_stats = draft.groupby('round').agg({
    'grade_score': ['mean', 'std'],
    'grade_score_uncertainty': 'mean',
    'pick_no': 'count'
}).reset_index()
round_stats.columns = ['round', 'avg_grade', 'std_grade', 'avg_uncertainty', 'count']

# Box plot with uncertainty
fig = go.Figure()

for round_num in sorted(draft['round'].unique())[:10]:  # First 10 rounds
    round_data = draft[draft['round'] == round_num]
    
    fig.add_trace(go.Box(
        y=round_data['grade_score'],
        name=f"Rd {round_num}",
        boxmean='sd',  # Show standard deviation
        marker_color=px.colors.sequential.Viridis[min(round_num-1, 9)]
    ))

fig.update_layout(
    title='Grade Score Distribution by Round (with std dev)',
    yaxis_title='Grade Score (0-100)',
    xaxis_title='Round',
    height=500,
    showlegend=False,
    template='plotly_white'
)

fig.show()

print("\nAverage Grade and Uncertainty by Round:")
print("=" * 60)
print(round_stats[round_stats['round'] <= 10])

## 4. Uncertainty vs Consistency Relationship

Players with high weekly variance should have wider confidence intervals.

In [ ]:
# Scatter: VOR CV vs Uncertainty Range
fig = px.scatter(
    draft[draft['vor_cv'].notna()],
    x='vor_cv',
    y='vor_uncertainty_range',
    color='consistency_tier',
    size='games_played',
    hover_data=['player_name', 'position', 'risk_tier'],
    title='VOR Coefficient of Variation vs Uncertainty Range',
    labels={'vor_cv': 'VOR Coefficient of Variation (relative uncertainty)',
            'vor_uncertainty_range': 'VOR Uncertainty Range (absolute)'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.add_annotation(
    text="High CV + High Range = Very Uncertain",
    xref="paper", yref="paper",
    x=0.95, y=0.95, showarrow=False,
    bgcolor="lightyellow"
)

fig.update_layout(height=600)
fig.show()

print("\nCorrelation between consistency and uncertainty:")
print("=" * 60)
corr = draft[['vor_cv', 'vor_uncertainty_range']].corr()
print(corr)

## 5. Key Insights & Recommendations

What does uncertainty tell us about draft decision-making?

In [ ]:
print("""
Uncertainty Quantification: Key Findings
═══════════════════════════════════════════════════════════════

1. PREVENTS FALSE PRECISION
   - Reporting "Grade: 73.2" implies more certainty than warranted
   - Better: "Grade: 73 ± 8" (range: 65-81)
   - Shows statistical humility and honesty

2. IDENTIFIES HIGH-RISK PICKS
   - Wide confidence intervals = volatile performance expected
   - Players with high weekly CV have wide VOR ranges
   - Example: Boom-bust WRs have ±30 VOR uncertainty

3. DISTINGUISHES NOISE FROM SIGNAL
   - Narrow CI + high VOR = confident elite pick
   - Wide CI + high VOR = risky upside play
   - Narrow CI + low VOR = confirmed bust
   - Wide CI + low VOR = unknown (small sample)

4. ENABLES BETTER RISK MANAGEMENT
   - Use lower bound for conservative projections
   - Use upper bound for best-case scenarios
   - Draft "high floor" (narrow CI) vs "high ceiling" (wide CI but high upper bound)

5. MAKES ANALYSIS PUBLICATION-READY
   - Academic standard to report confidence intervals
   - Peer reviewers expect uncertainty quantification
   - Shows methodological sophistication

Next Steps:
-----------
- Use uncertainty in trade analysis (don't overpay for volatile players)
- Build portfolio optimization (diversify across uncertainty profiles)
- Create Monte Carlo simulations using CI bounds
- Develop Bayesian updating as season progresses
""")